In [2]:
print('Hello World')

Hello World


In [2]:
import pandas as pd

df = pd.read_csv(
    "s3://enterprise-retail-data-lake/raw/orders/orderswithwrongrecords.txt",
    storage_options={"anon": False}
)

df.head()

,order_id,customer_id,product_id,amount,order_date
0,1001,C101,P201,500,2026-05-01
1,1002,NaN,P205,1200,2026-05-01
2,1003,C103,P209,-100,2026-05-02
3,1004,C104,P202,700,invalid_date
4,1005,C105,P210,900,2026-05-03


In [3]:
import pandas as pd
import boto3


from io import StringIO

bucket = "enterprise-retail-data-lake"
key = "raw/orders/orderswithwrongrecords.txt"

s3_client = boto3.client("s3")
obj = s3_client.get_object(Bucket=bucket, Key=key)

csv_content = obj["Body"].read().decode("utf-8")

df = pd.read_csv(StringIO(csv_content))

df.head()

,order_id,customer_id,product_id,amount,order_date
0,1001,C101,P201,500,2026-05-01
1,1002,NaN,P205,1200,2026-05-01
2,1003,C103,P209,-100,2026-05-02
3,1004,C104,P202,700,invalid_date
4,1005,C105,P210,900,2026-05-03


In [ ]:
import pandas as pd

# Identify invalid rows:
# - customer_id missing
# - amount negative
# - order_date not parseable as a valid date

invalid_customer = df["customer_id"].isna()
negative_amount = df["amount"] < 0
invalid_order_date = pd.to_datetime(df["order_date"], errors="coerce").isna()

invalid_rows = df[invalid_customer | negative_amount | invalid_order_date]

invalid_rows.head()